<a href="https://colab.research.google.com/github/jonathan9970/jonathan9970.github.io/blob/main/Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pgzrun

BOARD_SIZE = 8
SQUARE = 80

WIDTH = BOARD_SIZE * SQUARE
HEIGHT = BOARD_SIZE * SQUARE + 60

LIGHT = (240, 217, 181)
DARK = (181, 136, 99)

turn = "white"
selected = None
legal_moves = [] # Initialize global list for legal moves

white_time = 300  # 5 minutes
black_time = 300

game_over = False
winner = ""

# -----------------------------------
# Board setup
# -----------------------------------
board = [
    ["br","bn","bb","bq","bk","bb","bn","br"],
    ["bp"]*8,
    [".." for _ in range(BOARD_SIZE)], # Corrected initialization for empty rows
    [".." for _ in range(BOARD_SIZE)],
    [".." for _ in range(BOARD_SIZE)],
    [".." for _ in range(BOARD_SIZE)],
    ["wp"]*8,
    ["wr","wn","wb","wq","wk","wb","wn","wr"]
]

# Load piece images
piece_images = {} # dict
for color in ["w","b"]:
    for ptype in ["p","r","n","b","q","k"]:
        key = color+ptype
        piece_images[key] = key

# -----------------------------------
# Draw board
# -----------------------------------

def draw_row(row_index, c1, c2):
    y = row_index * SQUARE
    for col in range(BOARD_SIZE):
        x = col * SQUARE
        color = c1 if col % 2 == 0 else c2
        rect = Rect(x, y, SQUARE, SQUARE)
        screen.draw.filled_rect(rect, color)

def draw_board():
    for row in range(BOARD_SIZE):
        if row % 2 == 0:
            draw_row(row, LIGHT, DARK)
        else:
            draw_row(row, DARK, LIGHT)

# -----------------------------------
# Draw pieces
# -----------------------------------

def draw_pieces():
    for r in range(BOARD_SIZE): # r is 0,1,2,3,4,5,6,7
        for c in range(BOARD_SIZE):
            piece = board[r][c]
            if piece != "..":
                screen.blit(piece, (c*SQUARE, r*SQUARE))

## Notice that: Screen coordinates use (x, y)
## x = horizontal = column, y = vertical = row

# -----------------------------------
# Draw selection
# -----------------------------------

def draw_selection():
    if selected:
        r, c = selected
        rect = Rect(c*SQUARE, r*SQUARE, SQUARE, SQUARE)
        screen.draw.rect(rect, (0,255,0)) ## highlight color green

# -----------------------------------
# Chess Logic
	# •	sr = start row
	# •	sc = start column
	# •	tr = target row
	# •	tc = target column
  #When determining movement direction on the board,
  # the values of step_r (row step) and step_c (column step)
  # indicate how the position changes at each step along the path.
  #1. moving to the right means there is no change in rows but an increase in columns,
  #so step_r = 0 and step_c = 1.
  #2. Moving to the left also keeps the row the same but decreases the column,
  #giving step_r = 0 and step_c = -1.
  #3. Moving down increases the row while keeping the column unchanged,
  # so step_r = 1 and step_c = 0,
  #4. moving up decreases the row with no column change,
  #resulting in step_r = -1 and step_c = 0.
  #5. For a diagonal movement such as down-right,
  # both the row and column increase together,
  # so step_r = 1 and step_c = 1.
  # These step values allow the program to advance square-by-square in the correct direction
  # when checking movement paths.
# -----------------------------------

def path_clear(sr, sc, tr, tc):
    dr = tr - sr
    dc = tc - sc
    steps = max(abs(dr), abs(dc)) # number of squares moved = largest distance
    step_r = (dr // steps) if dr != 0 else 0 #Possible outcomes: 1→moving down; -1→moving up; 0→no vertical movement
    step_c = (dc // steps) if dc != 0 else 0

    for i in range(1, steps): #check squares between start and end, skip destination square
        r = sr + i*step_r    #Compute intermediate square, walks along the path.
        c = sc + i*step_c
        if board[r][c] != "..": #if square contains a piece.
            return False #Path is blocked → move not allowed.
    return True
    #Can the piece at (start row, start column) reach the target square using its normal movement pattern
def can_piece_reach(sr, sc, tr, tc):
    piece = board[sr][sc]
    if piece == "..":
        return False

    p = piece[1] #strip out the color information w/b, and leaving only the chesspiece type
    dr = tr - sr #Calculate movement distance
    dc = tc - sc

    if p == "p":
        color = piece[0] #Get pawn color
        direction = -1 if color == "w" else 1 #White moves upward (row decreases), Black moves downward (row increases).
        # Pawn captures diagonally (abs(dc) == 1) or moves straight (dc == 0)
        if abs(dc) == 1 and dr == direction and board[tr][tc] != "..": # Capture
            return True
        if dc == 0 and dr == direction and board[tr][tc] == "..": # Single move forward
            return True
        if dc == 0 and dr == 2 * direction and sr == (6 if color == "w" else 1) and board[tr][tc] == ".." and board[sr + direction][sc] == "..": # Double move forward
            return True
        return False

    if p == "r":
        return (sr == tr or sc == tc) and path_clear(sr, sc, tr, tc)

    if p == "b":
        return abs(dr) == abs(dc) and path_clear(sr, sc, tr, tc)

    if p == "q":
        return (sr == tr or sc == tc or abs(dr) == abs(dc)) and path_clear(sr, sc, tr, tc)

    if p == "n":
        return (abs(dr), abs(dc)) in [(2,1),(1,2)]

    if p == "k":
        return max(abs(dr), abs(dc)) == 1

    return False

def is_square_attacked(row, col, by_color): #Is the square at (row, col) being attacked by any piece of a specific color?
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            piece = board[r][c]
            if piece != ".." and piece[0] == by_color:
              #by_color → which side is attacking, attacked by "by_color" pieces
                # Temporarily remove king from its position to prevent infinite recursion during check for castling safety
                original_target_piece = board[row][col]
                temp_piece = board[r][c]
                board[row][col] = ".."
                board[r][c] = ".."

                can_reach = can_piece_reach(r, c, row, col)

                board[r][c] = temp_piece
                board[row][col] = original_target_piece

                if can_reach:
                    return True #YES → square is attacked
    return False #NO pieces → square is safe

def find_king(color):
    king_piece = ("w" if color=="white" else "b") + "k"
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if board[r][c] == king_piece:
                return (r,c)
    return None

def is_in_check(color):
    king_pos = find_king(color)
    if king_pos is None:
        return False
    opponent = "b" if color == "white" else "w"
    return is_square_attacked(king_pos[0], king_pos[1], opponent)

def would_be_in_check(sr, sc, tr, tc, color):
  # Checks whether a move would leave your king in check before actually making the move permanently
  #color = which player we are checking (e.g., "w" or "b")

    original_piece_at_target = board[tr][tc]
    original_piece_at_start = board[sr][sc]

    # Temporarily make the move
    board[tr][tc] = board[sr][sc] #Moves the piece to target square.
    board[sr][sc] = ".." #Clears the starting square.

    in_check = is_in_check(color) # Check if king is in check

    #Undo the move, Restore original board state:
    board[sr][sc] = original_piece_at_start
    board[tr][tc] = original_piece_at_target
    #ensure no permanent changes were made.

    return in_check

# Global variables to track king and rook moved status for castling
white_king_moved = False
black_king_moved = False
white_kingside_rook_moved = False
white_queenside_rook_moved = False
black_kingside_rook_moved = False
black_queenside_rook_moved = False

def castle(sr, sc, tr, tc):
    global white_king_moved, black_king_moved, white_kingside_rook_moved, white_queenside_rook_moved, black_kingside_rook_moved, black_queenside_rook_moved

    piece = board[sr][sc]
    color = piece[0]
    king_start_row = 0 if color == "b" else 7
    opponent_color = "b" if color == "w" else "w"

    # Ensure it's the king trying to castle and it's at its starting square
    if piece[1] != "k" or sr != king_start_row or sc != 4:
        return False

    # Cannot castle if the king has moved or is currently in check
    if (color == "w" and white_king_moved) or (color == "b" and black_king_moved):
        return False

    current_turn_color = "white" if color == "w" else "black"
    if is_in_check(current_turn_color):
        return False

    # Kingside Castling (tc is sc + 2, target column 6)
    if tc == 6:
        rook_sc = 7
        # Check if kingside rook has moved
        if (color == "w" and white_kingside_rook_moved) or (color == "b" and black_kingside_rook_moved):
            return False

        # Check if rook is at its initial position and is the correct rook
        if board[sr][rook_sc] != color + "r":
            return False

        # Check if path is clear (squares between king and rook)
        if board[sr][5] != ".." or board[sr][6] != "..": # F and G files for white king, B and C for black
            return False

        # Check squares king moves through and lands on are not attacked
        if is_square_attacked(sr, 5, opponent_color) or is_square_attacked(sr, 6, opponent_color):
            return False

        # Perform the castling move (temporarily for validation, then actually move)
        original_king = board[sr][sc]
        original_rook = board[sr][rook_sc]
        board[sr][6] = original_king  # Move king
        board[sr][5] = original_rook  # Move rook
        board[sr][sc] = ".."
        board[sr][rook_sc] = ".."

        if is_in_check(current_turn_color): # Check if castling results in king being in check
            # Undo the move if it results in check
            board[sr][sc] = original_king
            board[sr][rook_sc] = original_rook
            board[sr][6] = ".."
            board[sr][5] = ".."
            return False

        # Revert changes for `valid_move` checking:
        board[sr][sc] = original_king
        board[sr][rook_sc] = original_rook
        board[sr][6] = ".."
        board[sr][5] = ".."

        return True

    # Queenside Castling (tc is sc - 2, target column 2)
    elif tc == 2:
        rook_sc = 0
        # Check if queenside rook has moved
        if (color == "w" and white_queenside_rook_moved) or (color == "b" and black_queenside_rook_moved):
            return False

        # Check if rook is at its initial position and is the correct rook
        if board[sr][rook_sc] != color + "r":
            return False

        # Check if path is clear (squares between king and rook)
        if board[sr][1] != ".." or board[sr][2] != ".." or board[sr][3] != "..":
            return False

        # Check squares king moves through and lands on are not attacked
        if is_square_attacked(sr, 3, opponent_color) or is_square_attacked(sr, 2, opponent_color):
            return False

        # Temporarily perform the move to check for check after castling
        original_king = board[sr][sc]
        original_rook = board[sr][rook_sc]
        board[sr][2] = original_king # Move king
        board[sr][3] = original_rook # Move rook
        board[sr][sc] = ".."
        board[sr][rook_sc] = ".."

        if is_in_check(current_turn_color):
            # Undo the move if it results in check
            board[sr][sc] = original_king
            board[sr][rook_sc] = original_rook
            board[sr][2] = ".."
            board[sr][3] = ".."
            return False

        # Revert changes for `valid_move` checking:
        board[sr][sc] = original_king
        board[sr][rook_sc] = original_rook
        board[sr][2] = ".."
        board[sr][3] = ".."

        return True

    return False

def valid_move(sr, sc, tr, tc):
    piece = board[sr][sc]
    target = board[tr][tc]

    if piece == "..":
        return False

    color = piece[0]
    p = piece[1]

    if target != ".." and target[0] == color: # If the destination square contains your own piece, move is illegal
        return False

    # Removed check for target being king, as it's implied by other rules (king can't capture king)

    dr = tr - sr
    dc = tc - sc
    is_legal = False

    # Check for castling attempt FIRST as it's a special king move
    if p == "k" and dr == 0 and abs(dc) == 2: # King moves two squares horizontally
        return castle(sr, sc, tr, tc) # Delegate castling validation to castle function

    if p == "p":
        direction = -1 if color=="w" else 1 # White pawns move upward (row decreases), Black pawns move downward (row increases).
        start_row = 6 if color=="w" else 1

        # Normal one-step move
        if dc==0 and dr==direction and target=="..":
            is_legal=True
        # Two-step move from start row
        elif dc==0 and sr==start_row and dr==2*direction \
         and board[sr+direction][sc]==".." and target=="..":
            is_legal=True
        # Capture move
        elif abs(dc)==1 and dr==direction and target!="..":
            is_legal=True

    elif p == "r":
        if (sr==tr or sc==tc) and path_clear(sr,sc,tr,tc):
            is_legal=True

    elif p == "b":
        if abs(dr)==abs(dc) and path_clear(sr,sc,tr,tc):
            is_legal=True

    elif p == "q":
        if (sr==tr or sc==tc or abs(dr)==abs(dc)) and path_clear(sr,sc,tr,tc):
            is_legal=True

    elif p == "n":
        if (abs(dr),abs(dc)) in [(2,1),(1,2)]:
            is_legal=True

    elif p == "k":
        # King regular move (one square in any direction)
        if max(abs(dr),abs(dc))==1:
            # A king cannot move into a square attacked by an opponent
            opponent = "b" if color=="w" else "w"
            if not is_square_attacked(tr,tc,opponent):
                is_legal=True

    if not is_legal:
        return False

    # Final check: would the move put own king in check?
    current_color_name = "white" if color=="w" else "black"
    if would_be_in_check(sr,sc,tr,tc,current_color_name):
        return False

    return True

def has_legal_moves(color):
    for sr in range(BOARD_SIZE):
        for sc in range(BOARD_SIZE):
            piece = board[sr][sc]
            if piece!=".." and ((color=="white" and piece[0]=="w") or (color=="black" and piece[0]=="b")):
                for tr in range(BOARD_SIZE):
                    for tc in range(BOARD_SIZE):
                        if valid_move(sr,sc,tr,tc):
                            return True
    return False

# -----------------------------------
# Mouse input
# -----------------------------------

def on_mouse_down(pos):
    global selected, turn, game_over, winner, legal_moves

    if game_over:
        return

    c = int(pos[0]//SQUARE)
    r = int(pos[1]//SQUARE)

    # Check if a piece is currently selected
    if selected is None:
        piece = board[r][c] # Retrieves the piece at the clicked square
        if piece!=".." and ((turn=="white" and piece[0]=="w") or (turn=="black" and piece[0]=="b")):
            selected=(r,c)
            legal_moves = [] # Clear previous legal moves
            # Populate legal_moves for the newly selected piece
            for tr in range(BOARD_SIZE):
                for tc in range(BOARD_SIZE):
                    # For castling moves, valid_move calls castle, which performs the move temporarily.
                    # We need to make sure to clean up the board state if valid_move is called for legal move generation.
                    # However, if castle returns True, it implies it's a valid castling move, and its internal logic
                    # already temporarily moves pieces and reverts them for check validation.
                    # So, we just check valid_move directly.
                    if valid_move(r, c, tr, tc):
                        legal_moves.append((tr, tc))
    else:
        sr,sc = selected
        if valid_move(sr,sc,r,c): # Checks if moving selected piece to target is legal

            # Handle castling (if it was a castling move, the `castle` function already performed the move and returned True)
            # We need to ensure that if valid_move returns true for castling, the actual board update happens here.
            # The `castle` function in its current form temporarily makes and reverts the move for validation only.
            # So, if valid_move returns true for a castling intent, we need to explicitly make the castling move here.

            piece_to_move = board[sr][sc]
            if piece_to_move[1] == 'k' and abs(c - sc) == 2: # This was a castling attempt
                color = piece_to_move[0]
                if c == 6: # Kingside castle
                    board[sr][6] = board[sr][sc] # Move king
                    board[sr][5] = board[sr][7] # Move rook
                    board[sr][sc] = ".."
                    board[sr][7] = ".."
                    if color == 'w': white_kingside_rook_moved = True
                    else: black_kingside_rook_moved = True
                elif c == 2: # Queenside castle
                    board[sr][2] = board[sr][sc] # Move king
                    board[sr][3] = board[sr][0] # Move rook
                    board[sr][sc] = ".."
                    board[sr][0] = ".."
                    if color == 'w': white_queenside_rook_moved = True
                    else: black_queenside_rook_moved = True

                # Mark king as moved for castling
                if color == 'w': white_king_moved = True
                else: black_king_moved = True

            else: # Not a castling move, perform standard move
                # Move piece
                board[r][c] = board[sr][sc]
                board[sr][sc] = ".."

                # Update king/rook moved status for normal king/rook moves
                moved_piece_type = board[r][c][1]
                moved_piece_color = board[r][c][0]

                if moved_piece_type == 'k':
                    if moved_piece_color == 'w':
                        white_king_moved = True
                    else:
                        black_king_moved = True
                elif moved_piece_type == 'r':
                    if moved_piece_color == 'w':
                        if sc == 0: white_queenside_rook_moved = True
                        elif sc == 7: white_kingside_rook_moved = True
                    else: # black rook
                        if sc == 0: black_queenside_rook_moved = True
                        elif sc == 7: black_kingside_rook_moved = True


            # Pawn Promotion
            moved_piece = board[r][c]
            if moved_piece == "wp" and r == 0:
                board[r][c] = "wq"
            elif moved_piece == "bp" and r == 7:
                board[r][c] = "bq"

            # Switch turns
            turn = "black" if turn=="white" else "white"
            legal_moves = [] # Clear legal moves after a successful move

            if not has_legal_moves(turn): #If player cannot make any legal move → game over
                game_over=True
                if is_in_check(turn):
                    winner = "black" if turn=="white" else "white"
                else:
                    winner="draw"
        else:
            legal_moves = [] # Clear legal moves if the click was invalid (clicked on an invalid target square)

        selected=None # After a move or invalid click, clear selection

# -----------------------------------
# Timer update
# -----------------------------------

def update(): # update() is called 60 times per second
    global white_time, black_time, game_over, winner

    if game_over:
        return

    if turn=="white":
        white_time-=1/60
        if white_time<=0:
            winner="black"
            game_over=True
    else:
        black_time-=1/60
        if black_time<=0:
            winner="white"
            game_over=True

# -----------------------------------
# Draw everything
# -----------------------------------

def draw():
    screen.clear()
    draw_board()

    # Draw legal move highlights *before* pieces
    for r, c in legal_moves:
        rect = Rect(c*SQUARE, r*SQUARE, SQUARE, SQUARE)
        screen.draw.filled_rect(rect, (0, 0, 255, 100)) # Blue with some transparency

    draw_selection()
    draw_pieces()

    screen.draw.text(f"Turn: {turn}", (10, HEIGHT-55), fontsize=30, color="white")
    screen.draw.text(f"W:{int(white_time)}  B:{int(black_time)}", (200, HEIGHT-55), fontsize=30, color="white")

    if not game_over and is_in_check(turn):
        screen.draw.text("CHECK!", (WIDTH-150, HEIGHT-55), fontsize=35, color="red")

    if game_over:
        if winner=="draw":
            screen.draw.text("STALEMATE!", center=(WIDTH/2, HEIGHT-30), fontsize=40, color="yellow")
        else:
            screen.draw.text(f"{winner.upper()} WINS!", center=(WIDTH/2, HEIGHT-30), fontsize=40, color="yellow")

pgzrun.go()
